In [1]:
from fastai.vision.all import *

img = TensorImage(torch.rand(3,224,224))

print(img.shape)

print("Total values:", img.numel())

torch.Size([3, 224, 224])
Total values: 150528


In [1]:

import torch
from fastai.vision.all import *

img = torch.tensor([
    [1.,2.,3.,4.,5.],
    [6.,7.,8.,9.,1.],
    [2.,3.,4.,5.,6.],
    [7.,8.,9.,1.,2.],
    [3.,4.,5.,6.,7.]
])

print(img)


tensor([[1., 2., 3., 4., 5.],
        [6., 7., 8., 9., 1.],
        [2., 3., 4., 5., 6.],
        [7., 8., 9., 1., 2.],
        [3., 4., 5., 6., 7.]])


In [1]:
import torch

image_patch = torch.tensor([
    [1.,2.,3.],
    [4.,5.,6.],
    [7.,8.,9.]
])

kernel = torch.tensor([
    [1.,0.,1.],
    [0.,1.,0.],
    [1.,0.,1.]
])

result = (image_patch * kernel).sum()

print(result)

tensor(25.)


In [1]:
import torch
import torch.nn.functional as F

image = torch.tensor([
    [1.,2.,3.,4.,5.],
    [6.,7.,8.,9.,1.],
    [2.,3.,4.,5.,6.],
    [7.,8.,9.,1.,2.],
    [3.,4.,5.,6.,7.]
])

kernel = torch.tensor([
    [1.,0.,1.],
    [0.,1.,0.],
    [1.,0.,1.]
])

image = image.unsqueeze(0).unsqueeze(0)
kernel = kernel.unsqueeze(0).unsqueeze(0)

feature_map = F.conv2d(image, kernel)

print(feature_map)

tensor([[[[17., 22., 27.],
          [33., 29., 25.],
          [22., 27., 23.]]]])


In [1]:
# multiple feature maps

import torch
import torch.nn as nn

conv = nn.Conv2d(
    in_channels=1,
    out_channels=3,
    kernel_size=3
)

print(conv.weight.shape)

torch.Size([3, 1, 3, 3])


In [1]:
# stride change the output size
import torch
import torch.nn as nn

x = torch.randn(1,1,7,7)

conv1 = nn.Conv2d(
    in_channels=1,
    out_channels=1,
    kernel_size=3,
    stride=1
)

conv2 = nn.Conv2d(
    in_channels=1,
    out_channels=1,
    kernel_size=3,
    stride=2
)

y1 = conv1(x)
y2 = conv2(x)

print("Stride 1:", y1.shape)
print("Stride 2:", y2.shape)

Stride 1: torch.Size([1, 1, 5, 5])
Stride 2: torch.Size([1, 1, 3, 3])


In [1]:
# padding reserve image size
import torch
import torch.nn as nn

x = torch.randn(1,1,5,5)

conv_no_pad = nn.Conv2d(
    in_channels=1,
    out_channels=1,
    kernel_size=3,
    padding=0
)

conv_pad = nn.Conv2d(
    in_channels=1,
    out_channels=1,
    kernel_size=3,
    padding=1
)

y1 = conv_no_pad(x)
y2 = conv_pad(x)

print("No padding :", y1.shape)
print("Padding=1  :", y2.shape)

No padding : torch.Size([1, 1, 3, 3])
Padding=1  : torch.Size([1, 1, 5, 5])


In [1]:
# ReLu in action
import torch
import torch.nn.functional as F

x = torch.tensor([
    -5.,
    -2.,
     0.,
     3.,
     7.
])

y = F.relu(x)

print(y)

tensor([0., 0., 0., 3., 7.])


In [1]:
# max pooling
import torch
import torch.nn.functional as F

x = torch.tensor([
    [[
        [1.,4.,2.,3.],
        [7.,2.,0.,1.],
        [5.,6.,8.,2.],
        [1.,3.,4.,5.]
    ]]
])

pooled = F.max_pool2d(x, kernel_size=2)

print(pooled)

tensor([[[[7., 3.],
          [6., 8.]]]])


In [1]:
# real CNN
import torch
import torch.nn as nn

model = nn.Sequential(
    nn.Conv2d(3,16,kernel_size=3,padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),

    nn.Conv2d(16,32,kernel_size=3,padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2)
)

x = torch.randn(1,3,224,224)

y = model(x)

print(y.shape)

torch.Size([1, 32, 56, 56])


In [1]:
# transfer learning
from fastai.vision.all import *

path = untar_data(URLs.PETS)

dls = ImageDataLoaders.from_name_re(
    path,
    get_image_files(path/"images"),
    pat=r'(.+)_\d+.jpg',
    item_tfms=Resize(224)
)

learn = vision_learner(
    dls,
    resnet34,
    metrics=error_rate
)

print(type(learn.model))

<div><progress max="811706944" value="811712512"></progress> 100.00% [811712512/811706944 00:19&lt;00:00]</div>

Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 192MB/s] 


<class 'torch.nn.modules.container.Sequential'>


In [2]:
# how freezing and unfreezing works

from fastai.vision.all import *

path = untar_data(URLs.PETS)

dls = ImageDataLoaders.from_name_re(
    path,
    get_image_files(path/"images"),
    pat=r'(.+)_\d+.jpg',
    item_tfms=Resize(224)
)

learn = vision_learner(
    dls,
    resnet34,
    metrics=error_rate
)


print(learn.model)

learn.freeze()

print("Frozen")

learn.unfreeze()

print("Unfrozen")



<div><progress max="811706944" value="811712512"></progress> 100.00% [811712512/811706944 00:26&lt;00:00]</div>

Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 203MB/s]


Sequential(
  (0): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  